# 03 - A/B test: flexible free-cancellation rate
**Simulated experiment (seed 42) - designed and analysed here, not real guests.** 20,000 booking-flow users, 50/50. Treatment sees a flexible free-cancellation rate. Primary metric: booking conversion. Guardrails: cancellation rate among bookers, revenue per user.

Pre-registered rules: ship only if conversion improves significantly AND neither guardrail degrades significantly.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
rng = np.random.default_rng(42)

In [2]:
N = 20000
users = pd.DataFrame({'user_id': np.arange(1, N+1), 'variant': rng.choice(['control','treatment'], size=N, p=[0.5,0.5])})
users['booked'] = [rng.random() < (0.131 if v == 'treatment' else 0.120) for v in users['variant']]
users['cancelled'] = [bool(b) and (rng.random() < (0.41 if v == 'treatment' else 0.37)) for b, v in zip(users['booked'], users['variant'])]
users['revenue_per_user_eur'] = np.where(users['booked'] & ~users['cancelled'], 105.0 * 3.2, 0.0)
users.to_csv('../data/experiment_users_simulated.csv', index=False)
users.groupby('variant').size()

variant
control       9938
treatment    10062
dtype: int64

## SRM check first - if the split is broken, nothing below means anything

In [3]:
counts = users.groupby('variant').size().reindex(['control','treatment']).to_numpy()
chi2, srm_p = stats.chisquare(counts)
print(f'users: {counts}, SRM chi2 p = {srm_p:.3f} -> split is clean' if srm_p > 0.05 else 'SRM FAILED')

users: [ 9938 10062], SRM chi2 p = 0.381 -> split is clean


## Primary metric: conversion

In [4]:
g = users.groupby('variant')
conv = g['booked'].mean()
bk = g['booked'].sum().reindex(['control','treatment']).to_numpy()
ns = g.size().reindex(['control','treatment']).to_numpy()
_, p_conv = proportions_ztest(bk, ns)
print(f"control {conv['control']:.3f} vs treatment {conv['treatment']:.3f} | lift {(conv['treatment']/conv['control']-1)*100:.1f}% | p = {p_conv:.4f}")

control 0.120 vs treatment 0.132 | lift 9.7% | p = 0.0131


## Guardrail 1: cancellations among bookers

In [5]:
bookers = users[users['booked']]
cxl = bookers.groupby('variant')['cancelled'].mean()
cc = bookers.groupby('variant')['cancelled'].sum().reindex(['control','treatment']).to_numpy()
cn = bookers.groupby('variant').size().reindex(['control','treatment']).to_numpy()
_, p_cxl = proportions_ztest(cc, cn)
print(f"control {cxl['control']:.3f} vs treatment {cxl['treatment']:.3f} | p = {p_cxl:.4f}")

control 0.376 vs treatment 0.402 | p = 0.1822


The point estimate moved the wrong way (37.6% -> 40.2%) but p = 0.18: at this sample the guardrail test cannot separate a real increase from noise. That is a power problem, and it is quantifiable:

In [6]:
h = proportion_effectsize(0.41, 0.37)
need_bookers = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.8)
print(f'bookers needed per arm to detect 37% -> 41%: {need_bookers:.0f}')
print(f'at ~12% conversion that is ~{need_bookers/0.12:.0f} users per arm - the test ran with ~10,000')

bookers needed per arm to detect 37% -> 41%: 2333
at ~12% conversion that is ~19438 users per arm - the test ran with ~10,000


## Guardrail 2: revenue per user

In [7]:
rc = users[users['variant'] == 'control']['revenue_per_user_eur']
rt = users[users['variant'] == 'treatment']['revenue_per_user_eur']
_, p_rpu = stats.ttest_ind(rt, rc, equal_var=False)
print(f'control EUR {rc.mean():.2f} vs treatment EUR {rt.mean():.2f} | p = {p_rpu:.3f}')

control EUR 25.15 vs treatment EUR 26.45 | p = 0.307


## Decision
**Ship with monitoring.** Conversion lift +9.7% is significant (p = 0.013) and neither guardrail is significantly worse - but the cancellation guardrail is underpowered (needs ~19.4k users per arm, ran with ~10k), and its point estimate rose. Ship with a weekly cancellation monitor for 4 weeks and a pre-set stop rule, and re-run the guardrail readout at full power before making the rate permanent. Calling this a clean win, or a failure, would both be wrong on this evidence.